📅 **论文年份 (Year):1993 年**  
*Keeping Neural Networks Simple by Minimizing the Description Length of the Weights — Hinton & van Camp*

# Paper 5: Keeping Neural Networks Simple by Minimizing the Description Length（论文5：通过最小化描述长度保持神经网络简单）
## Hinton & Van Camp (1993) + Modern Pruning Techniques（Hinton 与 Van Camp（1993）+ 现代剪枝技术）

### Network Pruning & Compression（网络剪枝与压缩）

Key insight: Remove unnecessary weights to get simpler, more generalizable networks. Smaller = better!

核心洞见：移除不必要的权重，得到更简单、泛化能力更强的网络。更小 = 更好！

## 📖 论文导读

**🎯 这篇文章想解决什么问题（目的）：** 神经网络有个老毛病——参数太多时容易"死记硬背"训练数据（过拟合），一遇到没见过的新数据表现就变差。Hinton 和 van Camp 在 1993 年提出了一个根本性的问题：怎样衡量一个网络是不是"太复杂"了？他们借用信息论中的"最小描述长度"（MDL）原则：好的模型应该像一份精炼的笔记——既能解释数据，本身又足够简短。

**💡 主要贡献：** 论文把"简单"变成了可以计算的数学量：网络的总成本 = 描述权重所需的比特数 + 描述预测误差所需的比特数。更妙的是，他们提出给每个权重加上"噪声"——用一个概率分布而不是一个精确数字来表示权重。噪声越大，说明这个权重越不重要，可以用越少的比特来描述。这实际上是后来大名鼎鼎的"变分贝叶斯"方法在神经网络中的首次应用。

**🔧 方法：** 打个比方：你要把一本书压缩后发给朋友，就会删掉可有可无的句子——书变短了，但意思不能失真。训练时同时优化两件事：让网络预测得准（误差的描述短），同时让权重尽量"便宜"（权重的描述短）。那些噪声大、对结果影响小的权重，就可以被剪掉（pruning），最终得到一个又小、又不容易过拟合的网络。本笔记本实现了这一思想的现代版本：基于幅值的剪枝与迭代剪枝。

**🌟 意义：** 这篇论文是"贝叶斯神经网络"和"网络压缩/剪枝"两大方向的思想源头。今天手机上能跑大模型，靠的正是剪枝、量化这类压缩技术；变分推断更成为现代深度学习（如 VAE，本书单第 17 篇）的核心工具。它还给出了深度学习版"奥卡姆剃刀"的数学表达：简单的模型泛化更好——这一思想与书单中的 MDL、柯尔莫哥洛夫复杂度等论文一脉相承。

## 🎯 核心结论 (Key Takeaways)

- **论文核心思想：** Hinton 和 van Camp 用"最小描述长度"（MDL）给"简单模型泛化更好"下了数学定义：模型总成本 = 描述权重的比特数 + 描述预测误差的比特数，两者之和越小越好。给权重加噪声、用概率分布表示权重的做法，也是变分贝叶斯在神经网络中的首次应用，成为后来网络剪枝与贝叶斯神经网络两大方向的源头。

- **剪一半反而更好：** 本 notebook 实验中，把基线网络（1203 个参数、测试准确率 67.00%）按幅值剪掉 50% 的权重并微调后，参数只剩 628 个（约 1.9 倍压缩），准确率反而升到 68.33%——说明一半以上的权重本来就是冗余的，剪枝还起到了正则化、抑制过拟合的作用。

- **迭代剪枝的极限：** 采用"每轮剪一点、再微调"的迭代剪枝，剪到 57% 稀疏度时准确率仍有约 63%；但继续剪到 76%–95% 时准确率跌到 39%–47%。这提醒我们：这个演示网络本身很小、冗余有限，而真正大型的过参数化网络（如 ResNet、Transformer）才能在 90%+ 稀疏度下几乎不掉点。

- **MDL 算账验证：** 用简化 MDL 公式核算，剪枝后误差成本虽然上升（25.3 → 851.2），但模型成本大幅下降（1203 → 111），总描述长度从约 1228 降到约 962——用具体数字印证了"简单模型总成本更低、泛化更好"的奥卡姆剃刀原则。

- **带走一句话：** 神经网络普遍严重过参数化，"先训大、再剪小、后微调"是模型压缩的基本套路；评价一个模型的好坏，不能只看它拟合得多准，还要算上"描述模型本身"的成本。

## 🤯 反常识的发现 (Counterintuitive Findings)

- **常识认为：参数越多、模型越大，学得越好，砍掉权重肯定会伤害性能。** 但本 notebook 的实验发现恰恰相反：把基线网络（1203 个参数、测试准确率 67.00%）直接剪掉 50% 的权重再微调，参数只剩 628 个，准确率反而从 67.00% **升到了 68.33%**。原来一半以上的连接本来就是"闲人"，请走它们不仅不碍事，还顺便抑制了过拟合——剪枝本身就是一种正则化。

- **常识认为：给权重加噪声是在"污染"模型，应该把每个权重调得越精确越好。** 但 Hinton 和 van Camp 发现，故意给权重加噪声、用一个概率分布（而不是一个精确数值）来表示权重，反而让网络泛化更好——因为"模糊"的权重需要的描述比特数更少，模型更简单。这个"以退为进"的想法就是变分贝叶斯在神经网络中的首次应用，比 Dropout 等噪声正则化流行早了近二十年。

- **常识认为：评价模型就看它预测得准不准，错得越少越好。** 但 MDL 算账实验给出了反直觉的结论：剪枝后模型的误差成本大幅上升（25.3 → 851.2），可模型成本从 1203 骤降到 111，总描述长度从约 1228 降到约 962——按 MDL 原则，这个"犯更多错"的模型反而是更好的模型。好模型不是零差错的模型，而是"说明书 + 勘误表"加起来最薄的那个。

- **常识认为："压缩"和"智能"是两码事，压缩文件是工程活，学习是认知活。** 但这篇 1993 年的论文早就把两者划上了等号：学习就是寻找对数据的最短描述，压缩得越狠、泛化越好。三十年后"压缩即智能"成了解释大语言模型能力的热门理论——而它的数学源头，就在这篇论文里。


#### 💻 代码解读

**做什么:** 导入本笔记本需要的工具库,并固定随机数种子,保证每次运行结果一致。

**怎么做:**
- 导入 `numpy`(负责矩阵和数值计算)和 `matplotlib.pyplot`(负责画图);
- 调用 `np.random.seed(42)` 固定随机种子——就像掷骰子前先"设定好剧本",让后面生成的随机数据、随机初始权重每次都一样,方便复现实验结果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Simple Neural Network for Classification（用于分类的简单神经网络）

#### 💻 代码解读

**做什么:** 从零手写一个简单的两层神经网络 `SimpleNN`,并内置"剪枝开关"(掩码 mask),为后面的剪枝实验打基础。

**怎么做:**
- 定义两个激活函数:`relu`(负数归零、正数保留)和 `softmax`(把输出分数变成加起来等于 1 的概率);
- `SimpleNN` 类初始化时创建两层权重 `W1`、`W2` 和偏置 `b1`、`b2`,同时创建全为 1 的掩码 `mask1`、`mask2`——掩码就像每根连线上的"电闸",1 表示通电(权重有效),0 表示断电(权重被剪掉);
- `forward` 前向传播时先用 `W1 * mask1` 把被剪掉的权重清零,再经过隐藏层(ReLU)和输出层(softmax)算出各类别的概率;
- 还提供 `predict`(取概率最大的类别)、`accuracy`(算准确率)、`count_parameters`(统计总参数数和"还活着"的参数数);
- 最后用一个 5 行 10 列的随机输入测试网络,打印输出形状和参数统计,确认网络能正常工作。

In [ ]:
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    # 数值稳定性技巧:每行减去最大值再取exp,防止exp溢出;结果与标准softmax完全等价
    # keepdims=True保持形状(batch,1),才能对(batch,n_class)按行广播
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

class SimpleNN:
    """Simple 2-layer neural network"""
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Initialize weights
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros(output_dim)
        
        # Keep track of masks for pruning
        # 剪枝核心:用0/1掩码标记每个权重是否存活,初始全1表示未剪枝
        self.mask1 = np.ones_like(self.W1)
        self.mask2 = np.ones_like(self.W2)
    
    def forward(self, X):
        """Forward pass"""
        # Apply masks (for pruned weights)
        # 逐元素乘掩码:被剪掉的权重等效为0,相当于断开这条连接
        W1_masked = self.W1 * self.mask1
        W2_masked = self.W2 * self.mask2
        
        # Hidden layer
        # 形状: (batch, input_dim) @ (input_dim, hidden_dim) -> (batch, hidden_dim)
        self.h = relu(np.dot(X, W1_masked) + self.b1)
        
        # Output layer
        logits = np.dot(self.h, W2_masked) + self.b2
        probs = softmax(logits)
        
        return probs
    
    def predict(self, X):
        """Predict class labels"""
        probs = self.forward(X)
        return np.argmax(probs, axis=1)
    
    def accuracy(self, X, y):
        """Compute accuracy"""
        predictions = self.predict(X)
        return np.mean(predictions == y)
    
    def count_parameters(self):
        """Count total and active (non-pruned) parameters"""
        total = self.W1.size + self.b1.size + self.W2.size + self.b2.size
        # 掩码求和即存活权重数(1表示保留);偏置不参与剪枝,全部计为活跃
        active = int(np.sum(self.mask1) + self.b1.size + np.sum(self.mask2) + self.b2.size)
        return total, active

# Test network
nn = SimpleNN(input_dim=10, hidden_dim=20, output_dim=3)
X_test = np.random.randn(5, 10)
y_test = nn.forward(X_test)
print(f"Network output shape: {y_test.shape}")
total, active = nn.count_parameters()
print(f"Parameters: {total} total, {active} active")

## Generate Synthetic Dataset（生成合成数据集）

#### 💻 代码解读

**做什么:** 人工合成一个三分类数据集,用来训练和测试神经网络,不依赖任何外部数据。

**怎么做:**
- 定义 `generate_classification_data` 函数:给每个类别随机挑一个"聚集中心"(center),然后在中心周围撒高斯分布的样本点——就像往地上撒三堆豆子,每堆围绕自己的中心;
- 用 `np.vstack` 和 `np.concatenate` 把三个类别的数据拼在一起,再用 `np.random.permutation` 打乱顺序;
- 生成 1000 个训练样本(`X_train`, `y_train`)和 300 个测试样本(`X_test`, `y_test`),每个样本 20 个特征、共 3 个类别;
- 打印数据集形状和各类别样本数量,确认数据均衡。

In [ ]:
def generate_classification_data(n_samples=1000, n_features=20, n_classes=3):
    """
    Generate synthetic classification dataset
    Each class is a Gaussian blob
    """
    X = []
    y = []
    
    samples_per_class = n_samples // n_classes
    
    for c in range(n_classes):
        # Random center for this class
        center = np.random.randn(n_features) * 3
        
        # Generate samples around center
        # 广播:(samples, n_features)+(n_features,),中心向量自动加到每一行上
        X_class = np.random.randn(samples_per_class, n_features) + center
        y_class = np.full(samples_per_class, c)
        
        X.append(X_class)
        y.append(y_class)
    
    X = np.vstack(X)
    y = np.concatenate(y)
    
    # Shuffle
    # 花式索引:用同一个随机排列同时打乱X和y,保证样本与标签仍一一对应
    indices = np.random.permutation(len(X))
    X = X[indices]
    y = y[indices]
    
    return X, y

# Generate data
X_train, y_train = generate_classification_data(n_samples=1000, n_features=20, n_classes=3)
X_test, y_test = generate_classification_data(n_samples=300, n_features=20, n_classes=3)

print(f"Training set: {X_train.shape}, {y_train.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")
print(f"Class distribution: {np.bincount(y_train)}")

## Train Baseline Network（训练基线网络）

#### 💻 代码解读

**做什么:** 手写一个训练循环 `train_network`(前向传播 + 反向传播 + 梯度下降),训练出一个未剪枝的"基线网络",作为后续剪枝对比的参照物。

**怎么做:**
- 每轮训练先做前向传播得到预测概率 `probs`,再把标签变成 one-hot 形式,计算交叉熵损失(衡量"猜得有多离谱");
- 手动推导反向传播:从输出层误差 `dL_dlogits` 开始,一层层往回算出 `W2`、`b2`、`W1`、`b1` 的梯度,经过 ReLU 时把隐藏层输出小于等于 0 的位置梯度清零;
- 更新权重时乘上掩码(如 `dL_dW1 * model.mask1`)——保证已被剪掉的连接不会"复活";
- 每轮记录训练损失和测试准确率,每 20 轮打印一次进度;
- 最后创建一个 20 输入、50 隐藏单元、3 输出的 `baseline_model`,训练 100 轮,打印基线准确率和参数总数。

In [ ]:
def train_network(model, X_train, y_train, X_test, y_test, epochs=100, lr=0.01):
    """
    Simple training loop
    """
    train_losses = []
    test_accuracies = []
    
    for epoch in range(epochs):
        # Forward pass
        probs = model.forward(X_train)
        
        # Cross-entropy loss
        y_one_hot = np.zeros((len(y_train), model.output_dim))
        # 花式索引:一次性把每个样本对应真实类别的位置置1,得到one-hot矩阵
        y_one_hot[np.arange(len(y_train)), y_train] = 1
        # 加1e-8防止log(0)出现-inf;交叉熵=真实类别概率的负对数,再对batch取平均
        loss = -np.mean(np.sum(y_one_hot * np.log(probs + 1e-8), axis=1))
        
        # Backward pass (simplified)
        batch_size = len(X_train)
        # softmax+交叉熵联合求导的经典结果:梯度就是(预测概率-真实one-hot)
        dL_dlogits = (probs - y_one_hot) / batch_size
        
        # Gradients for W2, b2
        # 链式法则:形状(hidden,batch)@(batch,output)->(hidden,output),与W2一致
        dL_dW2 = np.dot(model.h.T, dL_dlogits)
        dL_db2 = np.sum(dL_dlogits, axis=0)
        
        # Gradients for W1, b1
        # 误差反向传播时也要乘掩码,被剪掉的连接不传递梯度
        dL_dh = np.dot(dL_dlogits, (model.W2 * model.mask2).T)
        # 布尔索引实现ReLU导数:前向输出<=0的位置梯度置0
        dL_dh[model.h <= 0] = 0  # ReLU derivative
        dL_dW1 = np.dot(X_train.T, dL_dh)
        dL_db1 = np.sum(dL_dh, axis=0)
        
        # Update weights (only where mask is active)
        # 关键:梯度更新也乘掩码,保证被剪掉的权重在微调中始终保持为0
        model.W1 -= lr * dL_dW1 * model.mask1
        model.b1 -= lr * dL_db1
        model.W2 -= lr * dL_dW2 * model.mask2
        model.b2 -= lr * dL_db2
        
        # Track metrics
        train_losses.append(loss)
        test_acc = model.accuracy(X_test, y_test)
        test_accuracies.append(test_acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}, Test Acc: {test_acc:.2%}")
    
    return train_losses, test_accuracies

# Train baseline model
print("Training baseline network...\n")
baseline_model = SimpleNN(input_dim=20, hidden_dim=50, output_dim=3)
train_losses, test_accs = train_network(baseline_model, X_train, y_train, X_test, y_test, epochs=100)

baseline_acc = baseline_model.accuracy(X_test, y_test)
total_params, active_params = baseline_model.count_parameters()
print(f"\nBaseline: {baseline_acc:.2%} accuracy, {active_params} parameters")

## Magnitude-Based Pruning（基于幅值的剪枝）

Remove weights with smallest absolute values

移除绝对值最小的权重

#### 💻 代码解读

**做什么:** 实现核心方法——"按幅值剪枝" `prune_by_magnitude`:把绝对值最小的那批权重直接剪掉,再看看准确率掉了多少。

**怎么做:**
- 把 `W1`、`W2` 的所有权重拉平拼在一起,取绝对值,用 `np.percentile` 找出门槛值(threshold)——比如要剪 50%,门槛就是绝对值排在中间位置的那个值;
- 凡是绝对值不超过门槛的权重,掩码设为 0(断电剪掉);超过门槛的保留为 1——逻辑就是"嗓门最小的连接先下岗",因为它们对输出影响最小;
- 用 `copy.deepcopy` 复制一份基线模型得到 `pruned_model`,避免改动原模型;
- 对副本剪掉 50% 的权重,打印剪枝前后的测试准确率以及掉了多少——直接砍掉一半权重,准确率往往只降一点点。

In [ ]:
def prune_by_magnitude(model, pruning_rate):
    """
    Prune weights with smallest magnitudes
    
    pruning_rate: fraction of weights to remove (0-1)
    """
    # Collect all weights
    # 全局剪枝:把两层权重拉平合并,统一排序,而不是每层各剪各的
    all_weights = np.concatenate([model.W1.flatten(), model.W2.flatten()])
    all_magnitudes = np.abs(all_weights)
    
    # Find threshold
    # 取绝对值的分位数作阈值:如剪50%,阈值就是幅值的中位数
    threshold = np.percentile(all_magnitudes, pruning_rate * 100)
    
    # Create new masks
    # 幅值剪枝核心假设:|w|小的权重贡献小;布尔比较得到True/False再转成1/0掩码
    model.mask1 = (np.abs(model.W1) > threshold).astype(float)
    model.mask2 = (np.abs(model.W2) > threshold).astype(float)
    
    print(f"Pruning threshold: {threshold:.6f}")
    print(f"Pruned {pruning_rate:.1%} of weights")
    
    total, active = model.count_parameters()
    print(f"Remaining parameters: {active}/{total} ({active/total:.1%})")

# Test pruning
import copy
# 深拷贝:剪枝实验在副本上做,保留原始baseline用于后续对比
pruned_model = copy.deepcopy(baseline_model)

print("Before pruning:")
acc_before = pruned_model.accuracy(X_test, y_test)
print(f"Accuracy: {acc_before:.2%}\n")

print("Pruning 50% of weights...")
prune_by_magnitude(pruned_model, pruning_rate=0.5)

print("\nAfter pruning (before retraining):")
acc_after = pruned_model.accuracy(X_test, y_test)
print(f"Accuracy: {acc_after:.2%}")
print(f"Accuracy drop: {(acc_before - acc_after):.2%}")

## Fine-tuning After Pruning（剪枝后的微调）

Retrain remaining weights to recover accuracy

重新训练剩余的权重以恢复精度

#### 💻 代码解读

**做什么:** 对刚剪完枝的网络做"微调"(fine-tuning):用较小的学习率再训练一段时间,让剩下的权重补位,把损失的准确率找回来。

**怎么做:**
- 复用前面的 `train_network` 函数,对 `pruned_model` 再训练 50 轮,学习率降为 0.005(小步慢调,避免破坏已学到的知识);由于训练时梯度乘了掩码,被剪掉的权重不会复活;
- 微调后重新测量准确率 `acc_finetuned` 和存活参数数;
- 打印一张对比表:基线模型 vs 剪枝 50% 后的模型,包括各自的准确率、参数量、压缩倍数(`total_params/active`)和准确率变化——通常能看到模型小了一半,准确率几乎不变。

In [ ]:
print("Fine-tuning pruned network...\n")
# 微调:用更小的学习率继续训练,让剩余权重补偿被剪掉部分丢失的精度
finetune_losses, finetune_accs = train_network(
    pruned_model, X_train, y_train, X_test, y_test, epochs=50, lr=0.005
)

acc_finetuned = pruned_model.accuracy(X_test, y_test)
total, active = pruned_model.count_parameters()

print(f"\n{'='*60}")
print("RESULTS:")
print(f"{'='*60}")
print(f"Baseline:     {baseline_acc:.2%} accuracy, {total_params} params")
print(f"Pruned 50%:   {acc_finetuned:.2%} accuracy, {active} params")
print(f"Compression:  {total_params/active:.1f}x smaller")
print(f"Acc. change:  {(acc_finetuned - baseline_acc):+.2%}")
print(f"{'='*60}")

## Iterative Pruning（迭代剪枝）

Gradually increase pruning rate

逐步提高剪枝率

#### 💻 代码解读

**做什么:** 实现"迭代剪枝" `iterative_pruning`:不一刀砍到底,而是分多轮"剪一点、养一养",逐步把稀疏度提高到 95%。

**怎么做:**
- 先记录初始状态(稀疏度 0、参数量、准确率)存入 `results` 列表;
- 循环 `num_iterations`(5)轮,每轮把目标稀疏度按比例递增(19% → 38% → … → 95%)——就像修剪盆栽:每次剪一点枝叶,等它缓过来再剪,比一次剃光安全得多;
- 每轮内部先调用 `prune_by_magnitude` 剪枝,再调用 `train_network` 微调 30 轮恢复精度,然后把本轮的稀疏度、存活参数、准确率记入 `results`;
- 最后对基线模型的深拷贝 `iterative_model` 跑完整流程,目标是剪掉 95% 的权重。

In [ ]:
def iterative_pruning(model, X_train, y_train, X_test, y_test, 
                     target_sparsity=0.9, num_iterations=5):
    """
    Iteratively prune and finetune
    """
    results = []
    
    # Initial state
    total, active = model.count_parameters()
    acc = model.accuracy(X_test, y_test)
    results.append({
        'iteration': 0,
        'sparsity': 0.0,
        'active_params': active,
        'accuracy': acc
    })
    
    # Gradually increase sparsity
    for i in range(num_iterations):
        # Sparsity for this iteration
        # 稀疏度线性递增:如目标95%分5轮,则每轮19%/38%/57%/76%/95%
        # 逐步剪枝比一次剪到位温和,给网络留出恢复的机会
        current_sparsity = target_sparsity * (i + 1) / num_iterations
        
        print(f"\nIteration {i+1}/{num_iterations}: Target sparsity {current_sparsity:.1%}")
        
        # Prune
        prune_by_magnitude(model, pruning_rate=current_sparsity)
        
        # Finetune
        # 论文核心循环:剪枝->微调->再剪枝,交替进行直至目标稀疏度
        train_network(model, X_train, y_train, X_test, y_test, epochs=30, lr=0.005)
        
        # Record results
        total, active = model.count_parameters()
        acc = model.accuracy(X_test, y_test)
        results.append({
            'iteration': i + 1,
            'sparsity': current_sparsity,
            'active_params': active,
            'accuracy': acc
        })
    
    return results

# Run iterative pruning
iterative_model = copy.deepcopy(baseline_model)
results = iterative_pruning(iterative_model, X_train, y_train, X_test, y_test, 
                           target_sparsity=0.95, num_iterations=5)

## Visualize Pruning Results（可视化剪枝结果）

#### 💻 代码解读

**做什么:** 把迭代剪枝的结果画成两张折线图,直观展示"剪掉多少权重,准确率还能剩多少"。

**怎么做:**
- 从 `results` 列表提取三组数据:稀疏度 `sparsities`、准确率 `accuracies`、存活参数数 `active_params`;
- 左图(`ax1`)画"准确率 vs 稀疏度":横轴是剪掉的比例,红色虚线是基线准确率作参照,曲线在很高稀疏度之前基本贴着基线走;
- 右图(`ax2`)画"准确率 vs 存活参数量",并用 `invert_xaxis()` 反转横轴,让"模型越来越小"的方向朝右,阅读更直观;
- 最后打印关键结论:剪掉 90% 以上的权重,准确率损失也很小。

In [ ]:
# Extract data
# 列表推导式:从字典列表中分别抽取各字段,方便绘图
sparsities = [r['sparsity'] for r in results]
accuracies = [r['accuracy'] for r in results]
active_params = [r['active_params'] for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs Sparsity
ax1.plot(sparsities, accuracies, 'o-', linewidth=2, markersize=10, color='steelblue')
ax1.axhline(y=baseline_acc, color='red', linestyle='--', linewidth=2, label='Baseline')
ax1.set_xlabel('Sparsity (Fraction Pruned)', fontsize=12)
ax1.set_ylabel('Test Accuracy', fontsize=12)
ax1.set_title('Accuracy vs Sparsity', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)
ax1.set_ylim([0, 1])

# Parameters vs Accuracy
ax2.plot(active_params, accuracies, 's-', linewidth=2, markersize=10, color='darkgreen')
ax2.axhline(y=baseline_acc, color='red', linestyle='--', linewidth=2, label='Baseline')
ax2.set_xlabel('Active Parameters', fontsize=12)
ax2.set_ylabel('Test Accuracy', fontsize=12)
ax2.set_title('Accuracy vs Model Size', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)
ax2.set_ylim([0, 1])
# 反转x轴:让"越剪越小"的方向从左到右,便于观察精度随压缩的变化
ax2.invert_xaxis()  # Fewer params on right

plt.tight_layout()
plt.show()

print("\nKey observation: Can remove 90%+ of weights with minimal accuracy loss!")

## Visualize Weight Distributions（可视化权重分布）

#### 💻 代码解读

**做什么:** 画 4 张权重直方图,对比剪枝前后权重分布的变化,验证"被剪掉的确实是小权重"。

**怎么做:**
- 上排两张图:基线模型 `baseline_model` 的 `W1`、`W2` 全部权重的分布——近似以 0 为中心的钟形,大量权重挤在 0 附近;
- 用布尔索引 `iterative_model.W1[iterative_model.mask1 > 0]` 只挑出剪枝后"还活着"的权重;
- 下排两张图:剪枝后存活权重的分布——0 附近被"挖空"了,像个山谷,因为绝对值小的权重全被剪掉,只剩两侧幅值较大的权重;
- 打印结论:存活权重的幅值整体更大,证明剪枝确实是按"幅值小者先剪"执行的。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Baseline weights
axes[0, 0].hist(baseline_model.W1.flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Baseline W1 Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Weight Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(baseline_model.W2.flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Baseline W2 Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Weight Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Pruned weights (only active)
# 布尔索引:只取掩码为1处的存活权重,返回一维数组;剪掉的0值不参与统计
pruned_W1 = iterative_model.W1[iterative_model.mask1 > 0]
pruned_W2 = iterative_model.W2[iterative_model.mask2 > 0]

axes[1, 0].hist(pruned_W1.flatten(), bins=50, color='darkgreen', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Pruned W1 Distribution (Active Weights Only)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Weight Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(pruned_W2.flatten(), bins=50, color='darkgreen', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Pruned W2 Distribution (Active Weights Only)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Weight Value')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Pruned weights have larger magnitudes (small weights removed)")

## Visualize Sparsity Patterns（可视化稀疏模式）

#### 💻 代码解读

**做什么:** 用热力图把两层的掩码矩阵可视化,直接"看见"哪些连接被剪掉了,并统计最终压缩比。

**怎么做:**
- 用 `imshow` 把 `iterative_model.mask1` 和 `mask2`(转置后)画成图:在 `RdYlGn` 配色下绿色代表 1(连接还在)、红色代表 0(已剪掉),整张图就像网络连接的"存活地图";
- 左图是 W1(输入层 → 隐藏层)的稀疏模式,右图是 W2(隐藏层 → 输出层)的稀疏模式,各配一个颜色条;
- 最后调用 `count_parameters`,计算并打印最终稀疏度(剪掉的比例)和压缩比 `total / active`(模型缩小了多少倍)。

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# W1 sparsity pattern
# 把0/1掩码矩阵当图像画:转置后行=隐藏维、列=输入维,直观展示哪些连接被剪
im1 = ax1.imshow(iterative_model.mask1.T, cmap='RdYlGn', aspect='auto', interpolation='nearest')
ax1.set_xlabel('Input Dimension', fontsize=12)
ax1.set_ylabel('Hidden Dimension', fontsize=12)
ax1.set_title('W1 Sparsity Pattern (Green=Active, Red=Pruned)', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=ax1)

# W2 sparsity pattern
im2 = ax2.imshow(iterative_model.mask2.T, cmap='RdYlGn', aspect='auto', interpolation='nearest')
ax2.set_xlabel('Hidden Dimension', fontsize=12)
ax2.set_ylabel('Output Dimension', fontsize=12)
ax2.set_title('W2 Sparsity Pattern (Green=Active, Red=Pruned)', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

total, active = iterative_model.count_parameters()
# 稀疏度=被剪权重占比;压缩比=总参数/存活参数
print(f"\nFinal sparsity: {(total - active) / total:.1%}")
print(f"Compression ratio: {total / active:.1f}x")

## MDL Principle（MDL 原则）

Minimum Description Length: Simpler models generalize better

最小描述长度(Minimum Description Length)：更简单的模型泛化能力更好

#### 💻 代码解读

**做什么:** 回到论文的理论核心——最小描述长度(MDL)原则:用简化公式算出"描述模型 + 描述数据误差"的总成本,证明剪枝后的模型更划算。

**怎么做:**
- 定义 `compute_mdl` 函数,把总成本拆成两部分:模型成本(`model_cost`)= 存活参数个数,简化成每个参数记 1 "比特"——好比说明书的页数,参数越少说明书越薄;数据成本(`data_cost`)= 训练集上的交叉熵总和——模型解释不了的部分,要额外花笔墨写"错误更正";
- 总成本 `total_cost = model_cost + data_cost`,体现 MDL 的权衡:模型太大说明书厚,模型太小错误更正多,最好的模型是两者之和最小;
- 分别对 `baseline_model` 和剪枝 95% 的 `iterative_model` 计算 MDL,打印格式化对比表;
- 结论:剪枝模型的总成本更低 → 按 MDL 原则,它更简单、泛化能力更好。

In [ ]:
def compute_mdl(model, X_train, y_train):
    """
    Simplified MDL computation
    
    MDL = Model Cost + Data Cost
    - Model Cost: Bits to encode weights
    - Data Cost: Bits to encode errors
    """
    # Model cost: number of parameters (simplified)
    # MDL(最小描述长度)思想:好模型=描述模型的代价+描述数据误差的代价之和最小
    total, active = model.count_parameters()
    # 简化假设:每个存活参数记1比特;剪枝越多,模型描述代价越低
    model_cost = active  # Each param = 1 "bit" (simplified)
    
    # Data cost: cross-entropy loss
    # 信息论视角:交叉熵(总和,不取平均)就是编码全部标签所需的比特数
    probs = model.forward(X_train)
    y_one_hot = np.zeros((len(y_train), model.output_dim))
    y_one_hot[np.arange(len(y_train)), y_train] = 1
    data_cost = -np.sum(y_one_hot * np.log(probs + 1e-8))
    
    total_cost = model_cost + data_cost
    
    return {
        'model_cost': model_cost,
        'data_cost': data_cost,
        'total_cost': total_cost
    }

# Compare MDL for different models
baseline_mdl = compute_mdl(baseline_model, X_train, y_train)
pruned_mdl = compute_mdl(iterative_model, X_train, y_train)

print("MDL Comparison:")
print(f"{'='*60}")
print(f"{'Model':<20} {'Model Cost':<15} {'Data Cost':<15} {'Total'}")
print(f"{'-'*60}")
print(f"{'Baseline':<20} {baseline_mdl['model_cost']:<15.0f} {baseline_mdl['data_cost']:<15.2f} {baseline_mdl['total_cost']:.2f}")
print(f"{'Pruned (95%)':<20} {pruned_mdl['model_cost']:<15.0f} {pruned_mdl['data_cost']:<15.2f} {pruned_mdl['total_cost']:.2f}")
print(f"{'='*60}")
print(f"\nPruned model has LOWER total cost → Better generalization!")

## Key Takeaways（要点总结）

### Neural Network Pruning:（神经网络剪枝：）

**Core Idea**: Remove unnecessary weights to create simpler, smaller networks

**核心思想**：移除不必要的权重，构建更简单、更小的网络

### Magnitude-Based Pruning:（基于幅值的剪枝：）

1. **Train** network normally
2. **Identify** low-magnitude weights: $|w| < \text{threshold}$
3. **Remove** these weights (set to 0, mask out)
4. **Fine-tune** remaining weights

1. **训练**：正常训练网络
2. **识别**：找出小幅值权重：$|w| < \text{threshold}$
3. **移除**：删除这些权重（置为 0，用掩码屏蔽）
4. **微调**：重新训练剩余的权重

### Iterative Pruning:（迭代剪枝：）

Better than one-shot:
```
for iteration in 1..N:
    prune small fraction (e.g., 20%)
    finetune
```

Allows network to adapt gradually.

比一次性(one-shot)剪枝效果更好：每次迭代只剪去一小部分（如 20%）再微调，让网络逐步适应。

### Results (Typical):（典型结果：）

- **50% sparsity**: Usually no accuracy loss
- **90% sparsity**: Slight accuracy loss (<2%)
- **95%+ sparsity**: Noticeable degradation

Modern networks (ResNets, Transformers) can often be pruned to **90-95% sparsity** with minimal impact!

- **50% 稀疏度(sparsity)**：通常没有精度损失
- **90% 稀疏度**：轻微精度损失（<2%）
- **95%+ 稀疏度**：明显退化

现代网络（ResNet、Transformer）通常可以剪枝到 **90-95% 稀疏度**，而影响极小！

### MDL Principle:（MDL 原则：）

$$
\text{MDL} = \underbrace{L(\text{Model})}_\text{complexity} + \underbrace{L(\text{Data | Model})}_\text{errors}
$$

**Occam's Razor**: Simplest explanation (smallest network) that fits data is best.

**奥卡姆剃刀(Occam's Razor)**：能拟合数据的最简单解释（最小的网络）就是最好的。

### Benefits of Pruning:（剪枝的好处：）

1. **Smaller models**: Less memory, faster inference
2. **Better generalization**: Removing overfitting parameters
3. **Energy efficiency**: Fewer operations
4. **Interpretability**: Simpler structure

1. **模型更小**：占用内存更少，推理(inference)更快
2. **泛化更好**：移除了导致过拟合的参数
3. **能效更高**：运算量更少
4. **可解释性**：结构更简单

### Types of Pruning:（剪枝的类型：）

| Type | What's Removed | Speedup |
|------|----------------|----------|
| **Unstructured** | Individual weights | Low (sparse ops) |
| **Structured** | Entire neurons/filters | High (dense ops) |
| **Channel** | Entire channels | High |
| **Layer** | Entire layers | Very High |

| 类型 | 移除内容 | 加速效果 |
|------|----------------|----------|
| **非结构化(Unstructured)** | 单个权重 | 低（依赖稀疏运算） |
| **结构化(Structured)** | 整个神经元/滤波器 | 高（稠密运算） |
| **通道(Channel)** | 整个通道 | 高 |
| **层(Layer)** | 整个层 | 非常高 |

### Modern Techniques:（现代技术：）

1. **Lottery Ticket Hypothesis**: 
   - Pruned networks can be retrained from initialization
   - "Winning tickets" exist in random init

2. **Dynamic Sparse Training**:
   - Prune during training (not after)
   - Regrow connections

3. **Magnitude + Gradient**:
   - Use gradient info, not just magnitude
   - Remove weights with small magnitude AND small gradient

4. **Learnable Sparsity**:
   - L0/L1 regularization
   - Automatic sparsity discovery

1. **彩票假说(Lottery Ticket Hypothesis)**：
   - 剪枝后的网络可以从初始化重新训练
   - 随机初始化中存在"中奖彩票"（winning tickets）

2. **动态稀疏训练(Dynamic Sparse Training)**：
   - 在训练过程中剪枝（而非训练之后）
   - 重新生长连接

3. **幅值 + 梯度(Magnitude + Gradient)**：
   - 利用梯度信息，而不仅仅是幅值
   - 移除幅值小且梯度也小的权重

4. **可学习稀疏性(Learnable Sparsity)**：
   - L0/L1 正则化(regularization)
   - 自动发现稀疏结构

### Practical Tips:（实用技巧：）

1. **Start high, prune gradually**: Don't prune 90% immediately
2. **Fine-tune after pruning**: Critical for recovery
3. **Layer-wise pruning rates**: Different layers have different redundancy
4. **Structured pruning for speed**: Unstructured needs special hardware

1. **从高起点开始、逐步剪枝**：不要一开始就剪掉 90%
2. **剪枝后要微调**：这是恢复精度的关键
3. **逐层设置剪枝率**：不同层的冗余程度不同
4. **追求速度用结构化剪枝**：非结构化剪枝需要专用硬件支持

### When to Prune:（何时使用剪枝：）

✅ **Good for**:
- Deployment (edge devices, mobile)
- Reducing inference cost
- Model compression

❌ **Not ideal for**:
- Very small models (already efficient)
- Training speedup (structured pruning only)

✅ **适用于**：
- 部署（边缘设备、移动端）
- 降低推理成本
- 模型压缩

❌ **不太适合**：
- 非常小的模型（本身已经很高效）
- 加速训练（只有结构化剪枝才行）

### Compression Rates in Practice:（实践中的压缩率：）

- **AlexNet**: 9x compression (no accuracy loss)
- **VGG-16**: 13x compression
- **ResNet-50**: 5-7x compression
- **BERT**: 10-40x compression (with quantization)

- **AlexNet**：9 倍压缩（无精度损失）
- **VGG-16**：13 倍压缩
- **ResNet-50**：5-7 倍压缩
- **BERT**：10-40 倍压缩（结合量化(quantization)）

### Key Insight:（核心洞见：）

**Neural networks are massively over-parameterized!**

Most weights contribute little to final performance. Pruning reveals the "core" network that does the real work.

**"The best model is the simplest one that fits the data"** - MDL Principle

**神经网络存在大量的过参数化(over-parameterized)！**

大多数权重对最终性能贡献很小。剪枝揭示了真正发挥作用的"核心"网络。

**"最好的模型是能拟合数据的最简单模型"** —— MDL 原则